In [2]:
import sys
sys.path.append('../src')

from linguistic_brain_toolkit.utilities import get_path
PATH = get_path(1)

In [5]:
import sqlite3
import pandas as pd

from linguistic_brain_toolkit.data import read_data_from_database, load_dataframe_to_sqlite
from linguistic_brain_toolkit.nlp import apply_lemmatization_and_pos_tags
from linguistic_brain_toolkit.utilities import get_path

pd.set_option('display.precision', 15)  # Establecer la precisión a 15 decimales

PATH = get_path(1)
DATABASE_FOLDER = PATH + 'data/processed/transcriptions.db'

df = read_data_from_database(DATABASE_FOLDER, 'general')

df.head()

,start,stop,name,tier,source
0,0.0000000000000,0.0124716553288,,phone,adollshouse
1,0.0124716553288,0.3816326530610,sp,phone,adollshouse
2,0.3816326530610,0.4215419501130,AO2,phone,adollshouse
3,0.4215419501130,0.4514739229020,L,phone,adollshouse
4,0.4514739229020,0.5213151927440,R,phone,adollshouse


In [6]:
# Preprocesing
df['name'] = df['name'].str.lower()
df['tier'].value_counts()

df_graphemic_transcrip = df.query("tier == 'word'").copy()

# Filtrar el DataFrame para excluir las filas que contienen las cadenas buscadas
strings_to_exclude = ('{', 'sp', 'chuck-e-cheese')

condition = df_graphemic_transcrip.astype(str).apply(
    lambda row: row.str.contains('|'.join(strings_to_exclude)
                                 )).any(axis=1)
df_filtered = df_graphemic_transcrip[~condition].copy()

# df_filtered = df_filtered.sample(n=5000)  #AAA

# NLP

In [ ]:
# NLP
df_processed = apply_lemmatization_and_pos_tags(df_filtered, 'name', 'large')
del df, df_graphemic_transcrip, df_filtered  # Delete dataframes to free memory
df_processed.head()

In [ ]:
df_processed['pos_tag_1'].value_counts()

In [ ]:
df_processed['pos_tag_2'].value_counts()

In [ ]:
df_processed['lemmatized_text'].value_counts()

In [ ]:
cols_to_process = [col for col in df_processed.columns if col.startswith('pos_tag_')]

for col in cols_to_process:
    df_processed[col] = df_processed[col].fillna('')  # Manejar valores nulos

df_processed.reset_index(drop=True, inplace=True)
df_processed.drop(['pos_tags'], axis=1, inplace=True)  # Especificar axis=1 para eliminar columnas

load_dataframe_to_sqlite(
    df_processed,
    DATABASE_FOLDER,
    'nlp_graphemic_transcriptions',
    if_exists='replace',
    df_name='df_processed'
    )

# LLMs

In [ ]:
from langchain_ollama.llms import OllamaLLM

from linguistic_brain_toolkit.llms import LLMParallelProcessor

In [ ]:
# Approximation with GPT model
grammatical_category_prompt_template = """You are an English philologist.
Your task is to identify the grammatical category of the word '{word}'.
The format of your answer must be only the grammatical category, in English, to which the word belongs (without answering anything else or including line breaks or periods)."""

df_filtered['grammatical_category_formatted_prompt'] = grammatical_category_prompt_template

llm_gpt = OllamaLLM(model='gemma3:27b')

In [ ]:
# Approximation with reasoning model
# lexical_field_prompt_template = """Temporal prompt template"""

# df_filtered['lexical_field_formatted_prompt'] = lexical_field_prompt_template


tense_mark_prompt_template = """You are a philologist specializing in current English.
Your task will be to identify the word '{word}' contains linguistic marks (morphology or semantics) related to tense (past, present or future).
The output format should be two paragraphs:
The first paragraph should contain only the corresponding tense word or 'Does not apply' in case it does not correspond to any tense mark, i.e. only the values: 'past', 'present', 'future' and 'Does not apply'.
The second paragraph should contain the explanation of why the word has a tense mark."""

df_filtered['tense_mark_formatted_prompt'] = tense_mark_prompt_template

llm_rm = OllamaLLM(model='deepseek-r1:8b')

In [10]:
df_filtered.dropna(subset=['name'])
df_filtered.reset_index(drop=True, inplace=True)
df_filtered.head()

,start,stop,name,tier,source,grammatical_category_formatted_prompt,tense_mark_formatted_prompt
0,0.000000000000,0.0124716553288,,word,adollshouse,You are an English philologist.\nYour task is ...,You are a philologist specializing in current ...
1,0.381632653061,0.6609977324260,alright,word,adollshouse,You are an English philologist.\nYour task is ...,You are a philologist specializing in current ...
2,0.660997732426,0.9104308390020,thank,word,adollshouse,You are an English philologist.\nYour task is ...,You are a philologist specializing in current ...
3,0.910430839002,1.0102040816300,you,word,adollshouse,You are an English philologist.\nYour task is ...,You are a philologist specializing in current ...
4,1.010204081630,1.2496598639500,very,word,adollshouse,You are an English philologist.\nYour task is ...,You are a philologist specializing in current ...


In [11]:
print(df_filtered.shape)
df_filtered = df_filtered.sample(100).copy()

(148027, 7)


In [12]:
%%time
processor = LLMParallelProcessor(
    llm=llm_gpt,
    batch_size=30
)

# Procesar los lotes
processor.process_batches(
    df=df_filtered,
    data_column='name',  # Columna para formatear el prompt template
    prompt_column='grammatical_category_formatted_prompt',  # Columna con los prompts listos para ser formateados
    result_column='grammatical_category',  # Columna en la que se almacenarán los resultados del LLM
    data_placeholder='word'  # Variable en el prompt template a ser formateada
)

CPU times: total: 625 ms
Wall time: 2min 13s


In [ ]:
%%time
processor = LLMParallelProcessor(
    llm=llm_rm,
    batch_size=30
)

# Procesar los lotes
processor.process_batches(
    df=df_filtered,
    data_column='name',  # Columna para formatear el prompt template
    prompt_column='tense_mark_formatted_prompt',  # Columna con los prompts listos para ser formateados
    result_column='tense_mark',  # Columna en la que se almacenarán los resultados del LLM
    data_placeholder='word'  # Variable en el prompt template a ser formateada
)

CPU times: total: 12.3 s
Wall time: 2h 9min 26s


In [14]:
df_filtered['grammatical_category'] = df_filtered['grammatical_category'].str.rstrip('.')
df_filtered['grammatical_category'] = df_filtered['grammatical_category'].str.rstrip('\n')
df_filtered['grammatical_category'].value_counts()

grammatical_category
Preposition                  20
Verb                         13
Adverb                        9
Noun                          9
Article                       8
Pronoun                       8
Adjective                     5
Conjunction                   5
Contraction                   3
Subordinating conjunction     3
Interjection                  3
Possessive pronoun            3
Auxiliary verb                3
Numeral                       2
Interrogative adverb          1
Object Pronoun                1
Modal auxiliary               1
Demonstrative pronoun         1
Verb (present participle)     1
Adjectival noun               1
Name: count, dtype: int64

In [12]:
# df_filtered['grammatical_category'] = df_filtered['grammatical_category'].str.rstrip('.')
df_filtered['tense_mark'].value_counts()

tense_mark
<think>\nAlright, so I have this query where the user is asking about whether the word "i" in English shows any tense marks related to past, present, or future. Hmm, first off, I need to make sure I understand what they mean by "tense." Tense typically refers to grammatical categories indicating time, like past, present, and future.\n\nNow, looking at the word "i," it's a pronoun used for singular subjects. Pronouns usually don't carry tense information because they refer to subjects who may be in different tenses. For example, "I went" uses "went" for past tense, but the pronoun "I" itself doesn't change. So, does "i" have any morphological or semantic markers of tense? Morphologically, it doesn't have any inflections or changes related to tense because pronouns are invariant.\n\nSemantically, "i" is used in contexts that refer to actions happening now, past, or future based on the verbs associated with them. But the pronoun itself doesn't encode that information; the verb 

In [17]:
print(df_filtered.loc[124482, 'tense_mark'])

<think>
Alright, so I have this query where the user is asking about whether the word "i" in English shows any tense marks related to past, present, or future. Hmm, first off, I need to make sure I understand what they mean by "tense." Tense typically refers to grammatical categories indicating time, like past, present, and future.

Now, looking at the word "i," it's a pronoun used for singular subjects. Pronouns usually don't carry tense information because they refer to subjects who may be in different tenses. For example, "I went" uses "went" for past tense, but the pronoun "I" itself doesn't change. So, does "i" have any morphological or semantic markers of tense? Morphologically, it doesn't have any inflections or changes related to tense because pronouns are invariant.

Semantically, "i" is used in contexts that refer to actions happening now, past, or future based on the verbs associated with them. But the pronoun itself doesn't encode that information; the verb does. So, if som

In [13]:
df_filtered.drop([
    'grammatical_category_formatted_prompt', 'tense_mark_formatted_prompt'
    ], axis=1, inplace=True)  # Especificar axis=1 para eliminar columnas

load_dataframe_to_sqlite(
    df_filtered,
    DATABASE_FOLDER,
    'grammatical_category',
    if_exists='replace',
    df_name='df_filtered'
    )

DataFrame df_filtered loaded into table grammatical_category.


## tense with paralellism

In [ ]:
%%time
import re
import pandas as pd
from pydantic import BaseModel
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.pydantic import PydanticOutputParser
from langchain_ollama.llms import OllamaLLM

# 1) Esquema Pydantic para la salida
class TenseMark(BaseModel):
    tense: Literal['past', 'present', 'future', 'not apply']
    explanation: str

# 2) Función para limpiar bloques <think>…</think>
def strip_think_tags(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# 3) Instanciamos modelo y parser
llm = OllamaLLM(model='gemma3:4b')
parser = PydanticOutputParser(pydantic_object=TenseMark)

# 4) Definimos el prompt con las instrucciones JSON inyectadas
prompt = ChatPromptTemplate.from_template(
    """You are a philologist specializing in current English.
Your task is to identify whether the word '{word}' contains morphological
or semantic marks of tense (past, present, future).

{format_instructions}

word: {word}"""
)

# 5) Construimos la cadena base: prompt → LLM → parser
base_chain = prompt | llm | parser

# 6) Preparamos el DataFrame y la lista de inputs
df = df_filtered.copy().head(25)
words = df['name'].dropna().tolist()

# Cada input debe incluir 'word' y el bloque de instrucciones de formato
inputs = [
    {
        "word": w,
        "format_instructions": parser.get_format_instructions()
    }
    for w in words
]

# 7) Paralelizamos con el método oficial `batch` de LCEL
#    Esto lanza todas las llamadas LLM en paralelo y retorna una lista de TenseMark
raw_results = base_chain.batch(inputs)  # :contentReference[oaicite:0]{index=0}

# 8) Procesamos cada resultado: limpiamos <think> y serializamos
processed = [
    strip_think_tags(r.json() if isinstance(r, TenseMark) else TenseMark.parse_obj(r).json())
    for r in raw_results
]

# 9) Asignamos las respuestas validadas y limpias al DataFrame
df.loc[df['name'].notna(), 'grammatical_category'] = processed

# 10) (Opcional) Revisa el DataFrame resultante
df.head()

In [ ]:
print(df.loc[51982, 'grammatical_category'])